In [ ]:
# Imports
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.preprocessing import OrdinalEncoder
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
import optuna
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')



# Configurations
ROOT_PATH = "playground-series-s6e6"
EXTERNEL_DATA_PATH = "playground-series-s6e6/externel_dataset/star_classification.csv"
BENCHMARK = 0.97
ID = 'id'
TARGET = 'class'
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}
TARGET_INV_MAPPING = {
    0:"GALAXY",
    1:"QSO",
    2:"STAR"
}
N_FOLDS = 5
SEED = 42
XGB_OPTUNA = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


print(f"Reading Train | Test | Externel Data ... ")
train_df_org = pd.read_csv(os.path.join(ROOT_PATH, 'train.csv'))
test_df_org = pd.read_csv(os.path.join(ROOT_PATH, 'test.csv'))
sub_org = pd.read_csv(os.path.join(ROOT_PATH, 'sample_submission.csv'))
# EXTRA_DATA_PATH
extra_df_org = pd.read_csv(os.path.join(EXTERNEL_DATA_PATH))

print(f"Shap of train : {train_df_org.shape} | test : {test_df_org.shape} | submissions : {sub_org.shape} | extra : {extra_df_org.shape}")
print(f"Target distribution train : \n{train_df_org['class'].value_counts(normalize=True)} | extra : \n{extra_df_org['class'].value_counts(normalize=True)}")


c:\Users\admin\Desktop\kaggle_competition\predicting_stellar_class\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Feature Engineering

In [4]:
train_df = train_df_org.copy()
test_df = test_df_org.copy()
extr_df = extra_df_org.copy()

n_cols = train_df.drop(columns=[ID, TARGET], axis='columns').select_dtypes(exclude='object').columns.to_list()
c_cols = train_df.drop(columns=[ID, TARGET], axis='columns').select_dtypes(include='object').columns.to_list()
print(f"Numerical columns : {n_cols}")
print(f"Categorical columns : {c_cols}")

def engineer_astro_features(df, c_cols=c_cols):
    df = df.copy()
    c_cols = c_cols.copy()
    
    fltr_cols = ['u', 'g', 'r', 'i', 'z']
    df['dom_fltr_fe'] = df[fltr_cols].idxmin(axis=1)
    c_cols.append('dom_fltr_fe')
    agg_map = { 'mean': 'mean','max': 'max','min': 'min','std': 'std','med': 'median'}
    for suffix, func in agg_map.items():
        df[f'{suffix}_fltr_fe'] = getattr(df[fltr_cols], func)(axis=1)

    for i in range(len(fltr_cols)):
        for j in range(i + 1, len(fltr_cols)):
            b1, b2 = fltr_cols[i], fltr_cols[j]
            df[f'{b1}_{b2}'] = df[b1] - df[b2]
    
    # Redshift interaction terms (multiplication) across all 10 color index bands
    df['u_g_redshift'] = df['u_g'] * df['redshift']
    df['g_r_redshift'] = df['g_r'] * df['redshift']
    df['r_i_redshift'] = df['r_i'] * df['redshift']
    df['i_z_redshift'] = df['i_z'] * df['redshift']
    df['u_r_redshift'] = df['u_r'] * df['redshift']
    df['g_i_redshift'] = df['g_i'] * df['redshift']
    df['r_z_redshift'] = df['r_z'] * df['redshift']
    df['u_i_redshift'] = df['u_i'] * df['redshift']
    df['g_z_redshift'] = df['g_z'] * df['redshift']
    df['u_z_redshift'] = df['u_z'] * df['redshift']

    # Color curvature differences (differences of differences)
    df['ug_gr'] = df['u_g'] - df['g_r']
    df['ug_ri'] = df['u_g'] - df['r_i']
    df['ug_iz'] = df['u_g'] - df['i_z']
    df['gr_ri'] = df['g_r'] - df['r_i']
    df['gr_iz'] = df['g_r'] - df['i_z']
    df['ri_iz'] = df['r_i'] - df['i_z']
    
    # Discretized stellar color temperature bins
    df['stellar_color_bin_fe'] = pd.cut(
        df['g_r'],
        bins=[-np.inf, 0.0, 0.4, 0.8, 1.2, np.inf],
        labels=[0, 1, 2, 3, 4]
    ).fillna(2).astype(int)

    # Equatorial coordinates -> Cartesian coordinates
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])

    df['coord_x_fe'] = np.cos(delta_rad) * np.cos(alpha_rad)
    df['coord_y_fe'] = np.cos(delta_rad) * np.sin(alpha_rad)
    df['coord_z_fe'] = np.sin(delta_rad)

    # Redshift-scaled coordinates
    df['coord_x_redshift_fe'] = df['coord_x_fe'] * df['redshift']
    df['coord_y_redshift_fe'] = df['coord_y_fe'] * df['redshift']
    df['coord_z_redshift_fe'] = df['coord_z_fe'] * df['redshift']

    # Galactic coordinate transformation (J2000 NGP)
    x_gal = -0.05487554 * df['coord_x_fe'] - 0.87343710 * df['coord_y_fe'] - 0.48383499 * df['coord_z_fe']
    y_gal =  0.49410945 * df['coord_x_fe'] - 0.44482959 * df['coord_y_fe'] + 0.74698225 * df['coord_z_fe']
    z_gal = -0.86766614 * df['coord_x_fe'] - 0.19807639 * df['coord_y_fe'] + 0.45598380 * df['coord_z_fe']

    df['gal_x_fe'] = x_gal
    df['gal_y_fe'] = y_gal
    df['gal_z_fe'] = z_gal

    stats_df = df[fltr_cols]
    dom_vals = stats_df.to_numpy()[np.arange(len(df)), stats_df.columns.get_indexer(df['dom_fltr_fe'])]
    df['dom_redshift_magn_fe'] = dom_vals * df['redshift'].to_numpy()

    return df, c_cols
    

print(f"Shape before feature engineering: {train_df.shape} | {test_df.shape} | {extra_df_org.shape}")

train_df_fe, c_cols_fe = engineer_astro_features(train_df)
test_df_fe, c_cols_fe = engineer_astro_features(test_df)
extr_df_fe, c_cols_fe = engineer_astro_features(extr_df)

print(f"Shape after feature engineering: {train_df_fe.shape} | {test_df_fe.shape} | {extr_df_fe.shape}")

Numerical columns : ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
Categorical columns : ['spectral_type', 'galaxy_population']
Shape before feature engineering: (577347, 12) | (247435, 11) | (100000, 18)
Shape after feature engineering: (577347, 55) | (247435, 54) | (100000, 61)


In [5]:
# st_encoder = LabelEncoder()
# gp_encoder = LabelEncoder()
# fe_encoder = LabelEncoder()

# X = pd.concat([
#     train_df_fe.drop([ID, TARGET, 'spectral_type', 'galaxy_population'], axis=1),
#     test_df_fe.drop([ID, 'spectral_type', 'galaxy_population'], axis=1)
# ], axis=0).reset_index(drop=True)
# X['dom_fltr_fe'] = fe_encoder.fit_transform(X['dom_fltr_fe'])

# extr_X = extr_df_fe[X.columns]
# extr_X['dom_fltr_fe'] = fe_encoder.fit_transform(extr_df_fe['dom_fltr_fe'])

# st = pd.concat([
#     train_df_fe['spectral_type'],
#     test_df_fe['spectral_type']
# ], axis=0).reset_index(drop=True)

# gp = pd.concat([
#     train_df_fe['galaxy_population'],
#     test_df_fe['galaxy_population']
# ], axis=0).reset_index(drop=True)

# st_y = st_encoder.fit_transform(st)
# gp_y = gp_encoder.fit_transform(gp)

# # --------------------
# # Same split for both tasks
# # --------------------
# X_train, X_test, st_train, st_test, gp_train, gp_test = train_test_split(
#     X,
#     st_y,
#     gp_y,
#     test_size=0.2,
#     random_state=42,
#     stratify=st_y
# )

# # --------------------
# # Stage 1: Predict Galaxy Population
# # --------------------
# gp_model = LogisticRegression(max_iter=1000)

# print("Training galaxy_population model...")
# gp_model.fit(X_train, gp_train)

# gp_pred = gp_model.predict(X_test)
# gp_proba = gp_model.predict_proba(X_test)

# print(f"GP Accuracy: {accuracy_score(gp_test, gp_pred):.6f}")

# # --------------------
# # Add GP predictions as features
# # --------------------

# # Better: use probabilities instead of hard labels
# X_train_st = X_train.copy()
# X_test_st = X_test.copy()

# train_gp_proba = gp_model.predict_proba(X_train)
# test_gp_proba = gp_model.predict_proba(X_test)

# for i in range(train_gp_proba.shape[1]):
#     X_train_st[f'gp_prob_{i}'] = train_gp_proba[:, i]
#     X_test_st[f'gp_prob_{i}'] = test_gp_proba[:, i]

# # --------------------
# # Stage 2: Predict Spectral Type
# # --------------------
# st_model = LogisticRegression(max_iter=1000)

# print("Training spectral_type model...")
# st_model.fit(X_train_st, st_train)

# st_pred = st_model.predict(X_test_st)

# print(
#     f"ST Balanced Accuracy: "
#     f"{balanced_accuracy_score(st_test, st_pred):.6f}"
# )

In [ ]:
st_encoder = LabelEncoder()
gp_encoder = LabelEncoder()
fe_encoder = LabelEncoder()

X = pd.concat([
    train_df_fe.drop([ID, TARGET, 'spectral_type', 'galaxy_population'], axis=1),
    test_df_fe.drop([ID, 'spectral_type', 'galaxy_population'], axis=1)
], axis=0).reset_index(drop=True)
X['dom_fltr_fe'] = fe_encoder.fit_transform(X['dom_fltr_fe'])

extr_X = extr_df_fe[X.columns]
extr_X['dom_fltr_fe'] = fe_encoder.fit_transform(extr_df_fe['dom_fltr_fe'])

st = pd.concat([
    train_df_fe['spectral_type'],
    test_df_fe['spectral_type']
], axis=0).reset_index(drop=True)

gp = pd.concat([
    train_df_fe['galaxy_population'],
    test_df_fe['galaxy_population']
], axis=0).reset_index(drop=True)

st_y = st_encoder.fit_transform(st)
gp_y = gp_encoder.fit_transform(gp)

gp_model = LogisticRegression(max_iter=1000)
st_model = LogisticRegression(max_iter=1000)

# Training for galaxy population
print("Training galaxy_population model...")
gp_model.fit(X, gp_y)

gp_pred = gp_model.predict(extr_X)
gp_proba = gp_model.predict_proba(extr_X)


# Training for spectral type
X_train_st = X.copy()
train_gp_proba = gp_model.predict_proba(X_train_st)
for i in range(train_gp_proba.shape[1]):
    X_train_st[f'gp_prob_{i}'] = train_gp_proba[:, i]

for i in range(gp_proba.shape[1]):
    extr_X[f'gp_prob_{i}'] = gp_proba[:, i]

print("Training spectral_type model...")
st_model.fit(X_train_st, st_y)
st_pred = st_model.predict(extr_X)


extra_df_org['galaxy_population'] = gp_pred
extra_df_org['spectral_type'] = st_pred

extra_df_org['galaxy_population'] = gp_encoder.inverse_transform(extra_df_org['galaxy_population'])
extra_df_org['spectral_type'] = st_encoder.inverse_transform(extra_df_org['spectral_type'])
extra_df_org['id'] = list(range(train_df_org.shape[0], train_df_org.shape[0] + len(extra_df_org)))
extra_df_org = extra_df_org[train_df_org.columns]

extra_df_org.to_csv("playground-series-s6e6\\externel_dataset\\star_classification_pred_spectral_type_galaxy_pop_org.csv", index=False)

Training galaxy_population model...


Training spectral_type model...


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- gp_prob_0
- gp_prob_1
